# 3  - Sailkapena

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 1 - Datuak lortu

* **Breast Cancer Wisconsin**: *Tumore bat gaiztoa (malignant) ala onbera (benign) den iragartzea, zelulen neurri fisikoetan oinarrituz.*
   * Zuzenean *Scikit-learn*-etik kargatu daiteke `sklearn.datasets` modulua erabiliz
   * Eremuak dituen hiztegi gisako objektu bat:
      * `o.data` edo `o['data']`: aldagai iragarleak (*Numpy*-ko matrizea)
      * `o.target` edo `o['target']`: helburu-aldagaia (*Numpy*-ko arraya)
      * `o.feature_names` edo `o['feature_names']`: aldagai iragarleen izenak

In [ ]:
from sklearn import datasets

cancer = datasets.load_breast_cancer()
print(type(cancer))
#cancer
#cancer.data
#cancer['data']

Datuak DataFrame batetan jar ditzakegu:

In [ ]:
# DataFrame-a sortu aldagai iragarleekin
df = pd.DataFrame(data=cancer.data, columns=cancer.feature_names)

# Helburu-aldagaia gehitu (0: gaiztoa / malignant, 1: onbera / benign)
df['target'] = cancer.target

df

## 2 - Datuak aztertu

In [ ]:
df.info()

In [ ]:
df.hist(bins=50, figsize=(15,20));

## 3 - Train/Test banaketa egin

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.2, random_state=42)
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)
print(f'{df.shape = }')
print(f'{type(train) = }  {train.shape = }')
print(f'{type(test) = }  {test.shape = }')
#train
#test

## 4 - Aldagai iragarleak eta helburu-aldagaia ezberdindu 

In [ ]:
X_train = train.drop(columns=['target'])
y_train = train['target']
X_test = test.drop(columns=['target'])
y_test = test['target']
print(f'{type(X_train) = }')
print(f'{type(y_train) = }')
#X_train
#y_train
#X_test
#y_test

## 5 - Ereduak sortu, doitu eta ebaluatu

Atal honetan `ml_course` moduluak eskaintzen digun sailkapen eredu bat erabiliko dugu:
* `ACMEClassifier`: *A simple classification model to distinguish between different categories*

Modulua instalatu eta bi ereduak inportatu:

In [ ]:
!pip install git+https://github.com/mpenagar/ml_course.git
#import ml_course
#import importlib
#importlib.reload(ml_course)
from ml_course import ACMEClassifier

In [ ]:
?ACMEClassifier

In [ ]:
model = ACMEClassifier()
#model = ACMEClassifier(regularization_strength=10, max_iterations=10000, random_state=42)
model.fit(X_train, y_train)
y_train_predict = model.predict(X_train)
y_test_predict = model.predict(X_test)
#y_train_predict
#y_test_predict

### Zehaztasuna (*Accuracy*)

Sailkapen-eredu batek zuzen asmatutako iragarpenen proportzioa da. Ereduak aztertutako kasu guztietatik zenbatetan asmatu duen adierazten du ehuneko edo proportzio gisa:

$$\text{Accuracy} = \frac{\text{Iragarpen Zuzenak}}{\text{Iragarpen Guztiak}}$$

In [ ]:
guztiak = y_test.size
zuzenak = (y_test_predict == y_test).sum()
zehaztasuna = zuzenak / guztiak *100
print(f'{guztiak = }  {zuzenak = }  {zehaztasuna = :.2f}%')

*Scikit-learn*-ek zehaztasuna kalkulatzen duen `accuracy_score` eskaintzen digu:

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_train = accuracy_score(y_train, y_train_predict)
accuracy_test = accuracy_score(y_test, y_test_predict)
print(f"Ereduaren zehaztasuna - train: {accuracy_train*100:.2f}%  test: {accuracy_test*100:.2f}%")

*Scikit-learn*-en ereduek (eta antzekoak ere, gure kasuan `ml_course` moduluko `ACMEClassifier` ereduak) zehaztasuna kalkulatzen duen `score` metodoa eskaintzen dute:

```python
model = SomeClassificationModel(...)
model.fit(X_train, y_train)
accuracy = model.score(X_test, y_test)
```

In [ ]:
accuracy_train = model.score(X_train, y_train)
accuracy_test = model.score(X_test, y_test)
print(f"Ereduaren zehaztasuna - train: {accuracy_train*100:.2f}%  test: {accuracy_test*100:.2f}%")

## 6 - Ereduaren hiperparametroak doitzen

Aurreko gaiean ikusi genuen bezala, *Scikit-learn*-eko `GridSearchCV` erabil dezakegu:

In [ ]:
from sklearn.model_selection import GridSearchCV

grid = {
    'regularization_strength' : [0.01,0.1,1,10,100]
}

grid_search = GridSearchCV(
    estimator=ACMEClassifier(max_iterations=10000, random_state=42),
    param_grid=grid,
    cv = 5,
    scoring='accuracy'
)

grid_search.fit(X_train, y_train)

In [ ]:
print(f'{grid_search.best_params_ = } , {grid_search.best_score_ = }')

`GridSearchCV` exekutatzean agertzen diren *warning* guztiak aurrerago aztertuko ditugu. Oraingoz lortutako ereduaren errendimendua neurtuko dugu:

In [ ]:
model = grid_search.best_estimator_
accuracy_train = model.score(X_train, y_train)
accuracy_test = model.score(X_test, y_test)
print(f"Ereduaren zehaztasuna - train: {accuracy_train*100:.2f}%  test: {accuracy_test*100:.2f}%")

## 7 - Konbergentzia arazoak

`GridSearchCV` exekutatzean eredua askotan doitzen denez (denera 25 aldiz), eskuz jarri dugun `max_iter=10000` ez da nahikoa kasu guztietarako eta konbergentzia arazoak gertatzen dira. *Warning*-ak berak bi irtenbide gomendatzen dizkigu:

1. `max_iter` handitzea
   * Dagoeneko egin dugu, defektuzko balioa 100 zen.
   * Gehiago handitzen badugugu konbergentzira iritsi gintezke, baina seguraski gehiegi luzatuko dugu prozesua.
1. Datuak eskalatzea
   * Adibidez aurreko gaiko estandarizazioa erabiliz (`sklearn.preprocessing.StandardScaler`)
   * Arazoa datuen magnitudeak eragin badu, eskalatzeak konbergentziaren arazoa baztertuko du eta doikuntza azkarragoa izango da 
  
`StandardScaler`-en `transform()` metodoak *Numpy*-ko matrize bat bueltatuko badu ere, aldagai iragarle guztiak batera estandarizatu daitezke:

```python
scaler = StandardScaler()
scaler.fit(df)
df[:] = scaler.transform(df) # Dataframe baten zutabe guztiei Numpy-ko matrize bat esleitu
# beste aukera bat:
# df.iloc[:, :] = scaler.transform(df)
```

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler() 
scaler.fit(X_train)
print(f'{type(scaler.transform(X_train))=}  {scaler.transform(X_train).shape=}')
X_train[:] = scaler.transform(X_train)
X_test[:] = scaler.transform(X_test)

Saia gaitezen orain defektuzto `max_iter=100` erabiltzen (defektuzkoa denez, ez dugu jarriko):

In [ ]:
grid = {
    'regularization_strength' : [0.01,0.1,1,10,100]
}
grid_search = GridSearchCV(
    estimator=ACMEClassifier(random_state=42),
    param_grid=grid,
    cv = 5,
    scoring='accuracy'
)

grid_search.fit(X_train, y_train)
print(f'{grid_search.best_params_ = } , {grid_search.best_score_ = }')

Aurrekoa baina dexente bizkorrakoa, *warning*-ak desagertu dira eta bide batez errendimendua (5-fold CV-koa) ere hobetu dugu (96,26% &rarr; 97.14%)

Nahi izanez gero, `regularization_strength` hiperparametroaren balioa hobeto doitzen saia gintezke (`10` balioaren inguruan):

In [ ]:
grid = {
    'regularization_strength' : [2.5,5,10,20,40]
}
grid_search = GridSearchCV(
    estimator=ACMEClassifier(max_iterations=100, random_state=42),
    param_grid=grid,
    cv = 5,
    scoring='accuracy'
)

grid_search.fit(X_train, y_train)
print(f'{grid_search.best_params_ = } , {grid_search.best_score_ = }')

97.14% &rarr; 97.36% , 5-en inguruan jarraitu dezakedu:

In [ ]:
grid = {
    'regularization_strength' : [3,4,5,6,7]
}
grid_search = GridSearchCV(
    estimator=ACMEClassifier(max_iterations=100, random_state=42),
    param_grid=grid,
    cv = 5,
    scoring='accuracy'
)

grid_search.fit(X_train, y_train)
print(f'{grid_search.best_params_ = } , {grid_search.best_score_ = }')

Hobekuntzarik ez dagoenez, bilaketa amaitutzat eman eta ereduaren errendimendua neurtu dezakegu test azpimultzoan:

In [ ]:
model = grid_search.best_estimator_
accuracy_train = model.score(X_train, y_train)
accuracy_test = model.score(X_test, y_test)
print(f"Ereduaren zehaztasuna - train: {accuracy_train*100:.2f}%  test: {accuracy_test*100:.2f}%")

| System          | $\mathbf{Accuracy_{train}}$ | $\mathbf{Accuracy_{test}}$ |
|-----------------|---------------:|---------------:|
| **ACMEClassifier** |       98.68% |      98.25% |

## 8 - *Baseline* erreferentzia-eredua

Aurreko gaiean ikusi genuen bezala, ondo legoke erreferentziazko eredu inozo bat izatea. Aukerak:

1. Auzazki (`0`/`1`) erantzuten duen eredua
1. Beti berdina erantzuten duen eredua
   * `0` edo `1`? &rarr; Entrenamenduan gehienetan agertzen dena

### *Baseline* - Auzazko eredua

In [ ]:
np.random.seed(42)
y_train_predict = np.random.randint(0, 2, size=y_train.size)
y_test_predict = np.random.randint(0, 2, size=y_test.size)
accuracy_train = accuracy_score(y_train, y_train_predict)
accuracy_test = accuracy_score(y_test, y_test_predict)
print(f"Ereduaren zehaztasuna - train: {accuracy_train*100:.2f}%  test: {accuracy_test*100:.2f}%")

### *Baseline* - Moda eredua

Klase bakoitza zenbatetan agertzen den aztertu dezakegu:

In [ ]:
print(f'{(y_train==0).sum() = :d}')
print(f'{(y_train==1).sum() = :d}')

*Pandas*-eko serieek `moda()` metodoa definitua dute. Moda elementu bat baina gehiago izan daitekeenez, modaren emaitza beste Series bat izango da:

In [ ]:
s = pd.Series(['a','b','c','a','b','c','a','c'])
print(f'{type(s.mode()) = }')
s.mode()

Modetako elementuren bat lortzeko, beti lehenengoarekin gera gintezke, `s.mode()[0]`. Gure kasuan:

In [ ]:
y_mode = y_train.mode()[0]
print(f'{y_mode = :d}')
y_train_predict = np.full(y_train.shape, y_mode)
y_test_predict = np.full(y_test.shape, y_mode)
accuracy_train = accuracy_score(y_train, y_train_predict)
accuracy_test = accuracy_score(y_test, y_test_predict)
print(f"Ereduaren zehaztasuna - train: {accuracy_train*100:.2f}%  test: {accuracy_test*100:.2f}%")

Laburbilduz:

| System          | $\mathbf{Accuracy_{train}}$ | $\mathbf{Accuracy_{test}}$ |
|-----------------|---------------:|---------------:|
| **Baseline-rand** |        49.67% |      50.88% |
| **Baseline-mode** |        62.86% |      62.28% |
| **ACMEClassifier** |       98.68% |      98.25% |
